<a href="https://colab.research.google.com/github/2BerbyMarty2/search-algorithms-2026/blob/main/IDDFS_vs_BestFirstSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from matplotlib.artist import get
import heapq

In [2]:
def get_coords(node_id):
    # If it's a list, convert every ID in that list to a coordinate tuple
    if isinstance(node_id, list):
        return [(n // 6, n % 6) for n in node_id]

    # Otherwise, treat it as a single integer ID
    return (node_id // 6, node_id % 6)

def get_node(coords):
    if isinstance(coords, (int, float)):
        return int(coords)
    return coords[0] * 6 + coords[1]

In [3]:
def setup_maze():
  start_id = random.randint(0, 11)
  goal_id = random.randint(24, 35)

  all_nodes = list(range(36))
  valid_barriers = [n for n in all_nodes if n!= start_id and n!= goal_id]
  barriers = random.sample(valid_barriers, 4)

  return {
      'start': start_id,
      'goal': goal_id,
      'barriers': barriers
  }

In [4]:
def draw_maze(maze_nodes, path=None, title=None):
    fig, ax = plt.subplots(figsize=(5, 5))

    # Add title if provided
    if title:
        plt.title(title)

    # Grid Setup
    labels = range(6)
    centers = np.arange(0.5, 6.0)
    ax.set_xticks(centers)
    ax.set_yticks(centers)
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xticks(range(7), minor=True)
    ax.set_yticks(range(7), minor=True)
    ax.grid(True, which='minor', color='black', linestyle='-')
    ax.tick_params(which='major', length=0)
    ax.xaxis.tick_top()
    ax.invert_yaxis()
    ax.set_xlim(0, 6)
    ax.set_ylim(6, 0)

    # 1. Draw Start, Goal, and Barriers
    start_node = maze_nodes['start']
    goal_node = maze_nodes['goal']
    barriers = maze_nodes['barriers']

    ax.add_patch(patches.Rectangle(get_coords(start_node), 1, 1, color='green', alpha=0.8, label='Start'))
    ax.add_patch(patches.Rectangle(get_coords(goal_node), 1, 1, color='red', alpha=0.8, label='Goal'))

    for barrier_id in barriers:
        ax.add_patch(patches.Rectangle(get_coords(barrier_id), 1, 1, color='black', alpha=0.9))

    # 2. Draw Background Numbers
    for i in range(36):
        x, y = get_coords(i)

        # If box is colored, use white text; otherwise, faint black
        if i == start_node or i == goal_node or i in barriers:
            t_color, t_alpha = 'white', 0.9
        else:
            t_color, t_alpha = 'black', 0.2

        ax.text(x + 0.5, y + 0.5, str(i),
                va='center', ha='center',
                fontsize=10, color=t_color, weight='bold', alpha=t_alpha)

    # 3. Draw Path and Circulated Numbers
    if path:
        path_coords = [get_coords(node_id) for node_id in path]
        path_x = [c[0] + 0.5 for c in path_coords]
        path_y = [c[1] + 0.5 for c in path_coords]

        # Draw the blue line first (zorder 4)
        ax.plot(path_x, path_y, color='blue', linewidth=3, alpha=0.6, zorder=4, label='Path')

        # Circulate each number on the path
        for node_id in path:
            cx, cy = get_coords(node_id)
            center_x, center_y = cx + 0.5, cy + 0.5

            # Draw a white circle with a blue outline (zorder 5)
            circle = patches.Circle((center_x, center_y), 0.35,
                                    facecolor='white', edgecolor='blue',
                                    linewidth=2, zorder=5)
            ax.add_patch(circle)

            # Draw the number inside the circle (zorder 6)
            ax.text(center_x, center_y, str(node_id),
                    va='center', ha='center',
                    fontsize=11, color='black', weight='bold', zorder=6)

        # Directional arrows (zorder 3 so they sit behind circles)
        if len(path_x) > 1:
            ax.quiver(path_x[:-1], path_y[:-1],
                      np.diff(path_x), np.diff(path_y),
                      scale_units='xy', angles='xy', scale=1, color='blue', alpha=0.3, zorder=3)

    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.show()

In [5]:
def get_neighbors(node_id, barriers):
    # Rule (b): Convert current ID to coordinates to find neighbors
    x, y = get_coords(node_id)
    neighbors = []

    # Check all 8 directions (Horizontal, Vertical, Diagonal)
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue

            nx, ny = x + dx, y + dy

            # Rule (c): Stay within 6x6 grid and avoid barriers
            if 0 <= nx < 6 and 0 <= ny < 6:
                # Convert back to ID: ID = x*6 + y (based on your //6 and %6 logic)
                neighbor_id = nx * 6 + ny
                if neighbor_id not in barriers:
                    neighbors.append(neighbor_id)

    # Rule (a): Process neighbors in increasing order of ID
    neighbors.sort()
    return neighbors

In [6]:
def depth_limited_search(current_coords, goal_coords, barriers, depth_limit, total_visited, current_path):
    # Convert coordinates to IDs for processing and path tracking
    current_id = get_node(current_coords)
    goal_id = get_node(goal_coords)

    # Rule: Track every node visited for time complexity analysis
    total_visited.append(current_id)

    # Success: Goal reached
    if current_id == goal_id:
        return True, current_path + [current_id], total_visited

    # Failure: Depth limit reached
    if depth_limit <= 0:
        return False, None, total_visited

    # get_neighbors returns IDs sorted in increasing order
    for neighbor_id in get_neighbors(current_id, barriers):
        if neighbor_id not in current_path:
            # FIX: We must unpack THREE values here to match the function signature
            found, result_path, total_visited = depth_limited_search(
                get_coords(neighbor_id),
                goal_coords,
                barriers,
                depth_limit - 1,
                total_visited,
                current_path + [current_id]
            )

            if found:
                return True, result_path, total_visited

    return False, None, total_visited

In [7]:
def run_iddfs(maze_data):
    # Extract maze components
    start_node = maze_data['start']
    goal_node = maze_data['goal']
    barriers = maze_data['barriers']

    # Output variables
    visited_nodes = []          # All explored nodes (for time calculation)
    final_path = None
    result_found = False

    # Maximum depth = total nodes (6x6 grid = 36)
    max_depth = 36

    # Iterative Deepening Loop
    for depth_limit in range(max_depth):

        # Track nodes visited in this depth iteration
        depth_visited = []

        # Call Depth-Limited Search
        found, path, _ = depth_limited_search(
            start_node,
            goal_node,
            barriers,
            depth_limit,
            depth_visited,
            []
        )

        # Add this iteration's explored nodes to global list
        visited_nodes.extend(depth_visited)

        # If goal found → stop
        if found:
            final_path = path
            result_found = True
            print(f"Goal found at depth: {depth_limit}")
            break

    # Time = number of explored nodes (1 node = 1 minute)
    time_taken = len(visited_nodes)

    # Return structured output
    return {
        "result_found": result_found,
        "final_path": final_path,
        "visited_nodes": visited_nodes,
        "time_taken": time_taken
    }

In [8]:
import math

def chebyshev_distance(node_id, goal_id):
    x1, y1 = get_coords(node_id)
    x2, y2 = get_coords(goal_id)
    return max(abs(x1 - x2), abs(y1 - y2))

def heuristic_manhattan(node_id, goal_id):
    x1, y1 = get_coords(node_id)
    x2, y2 = get_coords(goal_id)
    # Sum of absolute differences (best for 4-way movement)
    return abs(x1 - x2) + abs(y1 - y2)

def heuristic_euclidean(node_id, goal_id):
    x1, y1 = get_coords(node_id)
    x2, y2 = get_coords(goal_id)
    # Straight-line distance
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def heuristic_octile(node_id, goal_id):
    x1, y1 = get_coords(node_id)
    x2, y2 = get_coords(goal_id)
    # Account for 1.0 cardinal and 1.414 diagonal costs
    dx = abs(x1 - x2)
    dy = abs(y1 - y2)
    return (dx + dy) + (math.sqrt(2) - 2) * min(dx, dy)

In [9]:
import heapq

def best_first_search(maze_data, heuristic_function):
    # Extract maze components
    start_node = maze_data['start']
    goal_node = maze_data['goal']
    barriers = maze_data['barriers']

    # Priority queue: (heuristic_cost, node)
    open_list = []

    # Use the passed heuristic function for the start node
    start_h = heuristic_function(start_node, goal_node)
    heapq.heappush(open_list, (start_h, start_node))

    # Tracking structures
    visited_nodes = []      # Order of exploration
    closed_set = set()      # Visited nodes
    parent = {}             # For path reconstruction

    result_found = False
    final_path = None

    # Main loop
    while open_list:
        heuristic_cost, current_node = heapq.heappop(open_list)

        # Skip if already processed
        if current_node in closed_set:
            continue

        # Mark visited
        closed_set.add(current_node)
        visited_nodes.append(current_node)

        # Goal check
        if current_node == goal_node:
            result_found = True

            # Reconstruct path
            path = []
            node = current_node
            while node in parent:
                path.append(node)
                node = parent[node]
            path.append(start_node)

            final_path = path[::-1]
            break

        # Explore neighbors
        for neighbor in sorted(get_neighbors(current_node, barriers)):
            if neighbor in closed_set:
                continue

            # Assign parent if not already assigned
            if neighbor not in parent:
                parent[neighbor] = current_node

            # Compute heuristic using the passed function
            h_value = heuristic_function(neighbor, goal_node)

            heapq.heappush(open_list, (h_value, neighbor))

    # Time calculation: 1 node = 1 minute
    time_taken = len(visited_nodes)

    return {
        "result_found": result_found,
        "final_path": final_path,
        "visited_nodes": visited_nodes,
        "time_taken": time_taken
    }

In [10]:
def calculate_path_cost(path):
    cost = 0
    for i in range(len(path) - 1):
        c1 = get_coords(path[i])
        c2 = get_coords(path[i+1])
        # Rule (e): Euclidean distance
        cost += np.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)
    return cost

In [11]:
import numpy as np

def run_comparison_task():
    num_trials = 3
    results = []

    # Storage for statistics calculation
    stats = {
        "IDDFS": {"times": [], "lengths": []},
        "BestFS": {"times": [], "lengths": []}
    }

    print(f"{'Maze':<5} | {'Algorithm':<10} | {'Nodes Explored':<15} | {'Time (min)':<10} | {'Path Cost':<10}")
    print("-" * 60)

    for i in range(1, num_trials + 1):
        # 1. Setup a unique maze for this trial
        maze = setup_maze()

        # 2. Run IDDFS
        iddfs_res = run_iddfs(maze)
        iddfs_cost = calculate_path_cost(iddfs_res['final_path']) if iddfs_res['final_path'] else 0

        # 3. Run Best-First Search
        bfs_res = best_first_search(maze, heuristic_euclidean)
        bfs_cost = calculate_path_cost(bfs_res['final_path']) if bfs_res['final_path'] else 0

        # Store for stats
        stats["IDDFS"]["times"].append(iddfs_res['time_taken'])
        stats["IDDFS"]["lengths"].append(iddfs_cost)
        stats["BestFS"]["times"].append(bfs_res['time_taken'])
        stats["BestFS"]["lengths"].append(bfs_cost)

        # Print table rows
        print(f"{i:<5} | {'IDDFS':<10} | {len(iddfs_res['visited_nodes']):<15} | {iddfs_res['time_taken']:<10} | {iddfs_cost:<10.2f}")
        print(f"{'':<5} | {'Best FS':<10} | {len(bfs_res['visited_nodes']):<15} | {bfs_res['time_taken']:<10} | {bfs_cost:<10.2f}")
        print("-" * 60)

    # 4. Calculate and Report Mean and Variance
    print("\n--- Statistical Analysis ---")
    for algo in ["IDDFS", "BestFS"]:
        m_time = np.mean(stats[algo]["times"])
        v_time = np.var(stats[algo]["times"])
        m_path = np.mean(stats[algo]["lengths"])
        v_path = np.var(stats[algo]["lengths"])

        print(f"\nResults for {algo}:")
        print(f"  Time taken -> Mean: {m_time:.2f}, Variance: {v_time:.2f}")
        print(f"  Path Length -> Mean: {m_path:.2f}, Variance: {v_path:.2f}")

# Execute the comparison
run_comparison_task()

Maze  | Algorithm  | Nodes Explored  | Time (min) | Path Cost 
------------------------------------------------------------
Goal found at depth: 4
1     | IDDFS      | 830             | 830        | 5.66      
      | Best FS    | 5               | 5          | 4.00      
------------------------------------------------------------
Goal found at depth: 5
2     | IDDFS      | 3458            | 3458       | 6.66      
      | Best FS    | 6               | 6          | 6.66      
------------------------------------------------------------
Goal found at depth: 5
3     | IDDFS      | 2395            | 2395       | 7.07      
      | Best FS    | 6               | 6          | 6.24      
------------------------------------------------------------

--- Statistical Analysis ---

Results for IDDFS:
  Time taken -> Mean: 2227.67, Variance: 1165064.22
  Path Length -> Mean: 6.46, Variance: 0.35

Results for BestFS:
  Time taken -> Mean: 5.67, Variance: 0.22
  Path Length -> Mean: 5.63, Varianc

In [12]:
def run_heuristic_experiment(heuristic_list):
    num_trials = 3

    for h_name, h_func in heuristic_list:
        print(f"\n{'='*25} EVALUATING: {h_name} {'='*25}")

        # Stats storage for this specific heuristic
        stats = {
            "IDDFS": {"times": [], "lengths": []},
            "BestFS": {"times": [], "lengths": []}
        }

        print(f"{'Maze':<5} | {'Algorithm':<12} | {'Nodes Explored':<15} | {'Time (min)':<10} | {'Path Cost':<10}")
        print("-" * 65)

        for i in range(1, num_trials + 1):
            # Generate a random maze
            maze = setup_maze()

            # Run IDDFS (Baseline - stays same regardless of heuristic)
            iddfs_res = run_iddfs(maze)
            iddfs_cost = calculate_path_cost(iddfs_res['final_path']) if iddfs_res['final_path'] else 0

            # Run Best-First Search with current heuristic
            bfs_res = best_first_search(maze, h_func)
            bfs_cost = calculate_path_cost(bfs_res['final_path']) if bfs_res['final_path'] else 0

            # Store stats
            stats["IDDFS"]["times"].append(iddfs_res['time_taken'])
            stats["IDDFS"]["lengths"].append(iddfs_cost)
            stats["BestFS"]["times"].append(bfs_res['time_taken'])
            stats["BestFS"]["lengths"].append(bfs_cost)

            # Print table
            print(f"{i:<5} | {'IDDFS':<12} | {len(iddfs_res['visited_nodes']):<15} | {iddfs_res['time_taken']:<10} | {iddfs_cost:<10.2f}")
            print(f"{'':<5} | {f'BestFS-{h_name[:3]}':<12} | {len(bfs_res['visited_nodes']):<15} | {bfs_res['time_taken']:<10} | {bfs_cost:<10.2f}")
            print("-" * 65)

        # Report stats for this heuristic
        print(f"\n--- Statistics for {h_name} ---")
        for algo in ["IDDFS", "BestFS"]:
            m_time = np.mean(stats[algo]["times"])
            v_time = np.var(stats[algo]["times"])
            m_path = np.mean(stats[algo]["lengths"])
            v_path = np.var(stats[algo]["lengths"])

            print(f"[{algo}] Time Avg: {m_time:.2f} (Var: {v_time:.2f}) | Path Avg: {m_path:.2f} (Var: {v_path:.2f})")

# --- 3. Execute the Full Test ---
heuristics_to_test = [
    ("Chebyshev", chebyshev_distance),
    ("Manhattan", heuristic_manhattan),
    ("Euclidean", heuristic_euclidean),
    ("Octile", heuristic_octile)
]

run_heuristic_experiment(heuristics_to_test)


========================= EVALUATING: Chebyshev =========================
Maze  | Algorithm    | Nodes Explored  | Time (min) | Path Cost 
-----------------------------------------------------------------
Goal found at depth: 4
1     | IDDFS        | 419             | 419        | 5.24      
      | BestFS-Che   | 5               | 5          | 5.24      
-----------------------------------------------------------------
Goal found at depth: 4
2     | IDDFS        | 299             | 299        | 4.41      
      | BestFS-Che   | 5               | 5          | 4.41      
-----------------------------------------------------------------
Goal found at depth: 3
3     | IDDFS        | 121             | 121        | 3.83      
      | BestFS-Che   | 4               | 4          | 3.83      
-----------------------------------------------------------------

--- Statistics for Chebyshev ---
[IDDFS] Time Avg: 279.67 (Var: 14987.56) | Path Avg: 4.50 (Var: 0.34)
[BestFS] Time Avg: 4.67 (Var: 0.2